In [1]:
from pathlib import Path
import sys
import pandas as pd

import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

from db import get_db_config, fetch_table_df, get_db

In [2]:
conn = get_db()
cur = conn.cursor()

In [13]:
# cur.execute("""commit;""")
# cur.fetchall()

In [15]:
# # delete all data from all tables and reset auto-incrementing ids
# cur.execute("""
# TRUNCATE TABLE 
# questions,
# users,
# subjects,
# topics,
# questions,
# reviews,
# badges,
# stats,
# settings,
# user_subjects,
# user_badges,
# sync_meta
# RESTART IDENTITY CASCADE;
# """)
# cur.execute("""commit;""")

In [16]:
# # schema info
# cur.execute("""
# SELECT
#     conrelid::regclass AS table_name,
#     conname AS constraint_name,
#     pg_get_constraintdef(oid) AS definition
# FROM pg_constraint
# WHERE contype IN ('p','u')
# ORDER BY table_name;
# """)

# tables = cur.fetchall()
# tables

In [51]:
import pandas as pd

# conn.rollback()
user_id = 3

overall_sql = """
SELECT
  COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0) AS attempts,
  COALESCE(SUM(s.correct), 0) AS correct,
  CASE
    WHEN COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0) = 0 THEN 0
    ELSE ROUND(
      100.0 * COALESCE(SUM(s.correct), 0) /
      COALESCE(SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)), 0)
    )
  END AS accuracy
FROM stats s
WHERE s.user_id = %s;
"""

topic_sql = """
WITH RECURSIVE
ranked_topics AS (
  SELECT
    id,
    name,
    subject_id,
    parent_topic_id,
    ROW_NUMBER() OVER (
      PARTITION BY parent_topic_id
      ORDER BY name
    ) AS rn
  FROM topics
),
topic_tree AS (
  SELECT
    id,
    name,
    subject_id,
    parent_topic_id,
    rn,
    CAST(rn AS TEXT) AS sort_path,
    0 AS level
  FROM ranked_topics
  WHERE parent_topic_id IS NULL
  UNION ALL
  SELECT
    c.id,
    c.name,
    c.subject_id,
    c.parent_topic_id,
    c.rn,
    tt.sort_path || '.' || c.rn,
    tt.level + 1
  FROM ranked_topics c
  JOIN topic_tree tt
    ON c.parent_topic_id = tt.id
),
descendants AS (
  SELECT id AS topic_id, id AS descendant_id
  FROM topics
  UNION ALL
  SELECT d.topic_id, t.id
  FROM descendants d
  JOIN topics t
    ON t.parent_topic_id = d.descendant_id
),
question_counts AS (
  SELECT
    topic_id,
    COUNT(*) AS total_questions
  FROM questions
  GROUP BY topic_id
),
practice_counts AS (
  SELECT
    q.topic_id AS topic_id,
    SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)) AS practiced,
    COALESCE(SUM(s.correct), 0) AS correct
  FROM stats s
  LEFT JOIN questions q
    ON q.id::text = s.question_id
  WHERE s.user_id = %s
    AND q.topic_id IS NOT NULL
  GROUP BY q.topic_id
),
topic_totals AS (
  SELECT
    d.topic_id,
    COALESCE(SUM(qc.total_questions), 0) AS total_questions,
    COALESCE(SUM(pc.practiced), 0) AS practiced,
    COALESCE(SUM(pc.correct), 0) AS correct
  FROM descendants d
  LEFT JOIN question_counts qc
    ON qc.topic_id = d.descendant_id
  LEFT JOIN practice_counts pc
    ON pc.topic_id = d.descendant_id
  GROUP BY d.topic_id
)
SELECT
  tt.id AS topic_id,
  tt.name AS topic_name,
  tt.subject_id AS subject_id,
  tt.parent_topic_id AS parent_topic_id,
  tt.level AS level,
  tt.sort_path AS sort_path,
  COALESCE(tot.total_questions, 0) AS total_questions,
  COALESCE(tot.practiced, 0) AS practiced,
  COALESCE(tot.correct, 0) AS correct,
  CASE
    WHEN COALESCE(tot.practiced, 0) = 0 THEN 0
    ELSE ROUND(100.0 * COALESCE(tot.correct, 0) / COALESCE(tot.practiced, 0))
  END AS progress
FROM topic_tree tt
LEFT JOIN topic_totals tot
  ON tot.topic_id = tt.id
ORDER BY tt.sort_path;
"""

subject_sql = """
WITH RECURSIVE
ranked_topics AS (
  SELECT
    id,
    name,
    subject_id,
    parent_topic_id,
    ROW_NUMBER() OVER (
      PARTITION BY parent_topic_id
      ORDER BY name
    ) AS rn
  FROM topics
),
topic_tree AS (
  SELECT
    id,
    name,
    subject_id,
    parent_topic_id,
    rn,
    CAST(rn AS TEXT) AS sort_path,
    0 AS level
  FROM ranked_topics
  WHERE parent_topic_id IS NULL
  UNION ALL
  SELECT
    c.id,
    c.name,
    c.subject_id,
    c.parent_topic_id,
    c.rn,
    tt.sort_path || '.' || c.rn,
    tt.level + 1
  FROM ranked_topics c
  JOIN topic_tree tt
    ON c.parent_topic_id = tt.id
),
descendants AS (
  SELECT id AS topic_id, id AS descendant_id
  FROM topics
  UNION ALL
  SELECT d.topic_id, t.id
  FROM descendants d
  JOIN topics t
    ON t.parent_topic_id = d.descendant_id
),
question_counts AS (
  SELECT
    topic_id,
    COUNT(*) AS total_questions
  FROM questions
  GROUP BY topic_id
),
practice_counts AS (
  SELECT
    q.topic_id AS topic_id,
    SUM(COALESCE(s.correct, 0) + COALESCE(s.wrong, 0)) AS practiced,
    COALESCE(SUM(s.correct), 0) AS correct
  FROM stats s
  LEFT JOIN questions q
    ON q.id::text = s.question_id
  WHERE s.user_id = %s
    AND q.topic_id IS NOT NULL
  GROUP BY q.topic_id
),
topic_totals AS (
  SELECT
    d.topic_id,
    COALESCE(SUM(qc.total_questions), 0) AS total_questions,
    COALESCE(SUM(pc.practiced), 0) AS practiced,
    COALESCE(SUM(pc.correct), 0) AS correct
  FROM descendants d
  LEFT JOIN question_counts qc
    ON qc.topic_id = d.descendant_id
  LEFT JOIN practice_counts pc
    ON pc.topic_id = d.descendant_id
  GROUP BY d.topic_id
),
topic_progress AS (
  SELECT
    tt.id AS topic_id,
    tt.name AS topic_name,
    tt.subject_id AS subject_id,
    tt.parent_topic_id AS parent_topic_id,
    COALESCE(tot.total_questions, 0) AS total_questions,
    COALESCE(tot.practiced, 0) AS practiced,
    COALESCE(tot.correct, 0) AS correct,
    CASE
      WHEN COALESCE(tot.practiced, 0) = 0 THEN 0
      ELSE ROUND(100.0 * COALESCE(tot.correct, 0) / COALESCE(tot.practiced, 0))
    END AS progress
  FROM topic_tree tt
  LEFT JOIN topic_totals tot
    ON tot.topic_id = tt.id
)
SELECT
  s.id AS subject_id,
  s.name AS subject_name,
  COALESCE(SUM(tp.total_questions), 0) AS total_questions,
  COALESCE(SUM(tp.practiced), 0) AS practiced,
  COALESCE(SUM(tp.correct), 0) AS correct,
  CASE
    WHEN COALESCE(SUM(tp.practiced), 0) = 0 THEN 0
    ELSE ROUND(100.0 * COALESCE(SUM(tp.correct), 0) / COALESCE(SUM(tp.practiced), 0))
  END AS progress
FROM subjects s
LEFT JOIN topic_progress tp
  ON tp.subject_id = s.id
 AND tp.parent_topic_id IS NULL
GROUP BY s.id, s.name
ORDER BY s.name;
"""

streak_sql = """
SELECT
  MAX(CASE WHEN key = 'streak_current_user_3' THEN value END) AS current_streak,
  MAX(CASE WHEN key = 'streak_longest_user_3' THEN value END) AS longest_streak,
  MAX(CASE WHEN key = 'streak_last_practice_date_user_3' THEN value END) AS last_practice_date
FROM settings
WHERE user_id = %s
  AND key IN (
    'streak_current_user_3',
    'streak_longest_user_3',
    'streak_last_practice_date_user_3'
  );
"""

def fetch_df(title, sql, params, columns=None):
    cur.execute(sql, params)
    rows = cur.fetchall()
    if columns is None:
        columns = [desc[0] for desc in cur.description]
    df = pd.DataFrame(rows, columns=columns)
    print(f"\n{title}")
    print(df.to_string(index=False))
    return df

overall = fetch_df(
    "Overall card",
    overall_sql,
    (user_id,),
    ["attempts", "correct", "accuracy"],
)

topics = fetch_df("Topic breakdown", topic_sql, (user_id,))
subjects = fetch_df("Subject breakdown", subject_sql, (user_id,))
streak = fetch_df(
    "Streak information",
    streak_sql,
    (user_id,),
    ["current_streak", "longest_streak", "last_practice_date"],
)



Overall card
 attempts  correct accuracy
       13       13      100

Topic breakdown
 topic_id            topic_name  subject_id  parent_topic_id  level sort_path total_questions practiced correct progress
        6              Addition           1              NaN      0         1               2         0       0        0
        8              Division           1              NaN      0         2               2         0       0        0
        9             Fractions           1              NaN      0         3               2         0       0        0
       18         Jumbled Words           2              NaN      0         4               4         0       0        0
       19       3 Letter Jumble           2             18.0      1       4.1               2         0       0        0
       20       5 Letter Jumble           2             18.0      1       4.2               2         0       0        0
        1 Multiplication Tables           1              NaN      

In [55]:
# Sample excute query
cur.execute("""
SELECT
  s.id AS stat_id,
  s.question_id,
  s.correct,
  s.wrong,
  s.practiced_at,
  q.id AS question_row_id,
  q.topic_id,
  t.name AS topic_name
FROM stats s
LEFT JOIN questions q
  ON q.id::text = s.question_id
LEFT JOIN topics t
  ON t.id = q.topic_id
WHERE s.user_id = 3
ORDER BY s.practiced_at DESC;


""")

tables = cur.fetchall()
pd.DataFrame(tables)

,0,1,2,3,4,5,6,7
0,12,31,1,0,2026-03-30 10:55:52.896,NaN,NaN,NaN
1,13,110,1,0,2026-03-30 10:50:35.753,NaN,NaN,NaN
2,11,19,1,0,2026-03-30 10:50:16.603,19.0,16.0,6 Letter Words
3,10,18,1,0,2026-03-30 10:44:00.402,18.0,15.0,5 Letter Words
4,9,17,1,0,2026-03-30 10:43:57.321,17.0,15.0,5 Letter Words
5,8,17,1,0,2026-03-30 10:41:20.294,17.0,15.0,5 Letter Words
6,7,16,1,0,2026-03-30 10:31:32.519,16.0,14.0,4 Letter Words
7,6,15,1,0,2026-03-30 10:31:01.122,15.0,14.0,4 Letter Words
8,5,14,1,0,2026-03-30 10:29:04.840,14.0,13.0,3 Letter Words
9,4,14,1,0,2026-03-30 10:18:49.874,14.0,13.0,3 Letter Words


In [3]:
# Sample excute query
cur.execute("""
SELECT user_id, key, value, updated_at
FROM settings
ORDER BY user_id, key;

""")

tables = cur.fetchall()
pd.DataFrame(tables)

,0,1,2,3
0,0,sync_mode,global_on,2026-04-05 18:55:39.000
1,1,active_device_key_user_1,device_bhavi_tab,2026-04-05 04:47:15.476
2,1,device_registry_user_1,"[{""backendKey"":""device_bhavi_tab"",""displayName...",2026-04-05 04:47:09.511
3,1,learn_progress_topic_2,"{""topicId"":2,""cardId"":13,""cardIndex"":2,""totalC...",2026-04-05 06:03:01.915
4,1,practice_session_topic_2,"{""practice"":{""stats"":{""attempts"":0,""correct"":0...",2026-04-05 06:03:07.554
5,2,active_device_key_user_2,device_mabhu_tab,2026-04-05 04:47:15.535
6,2,device_registry_user_2,"[{""backendKey"":""device_bhavi_tab"",""displayName...",2026-04-05 04:47:09.511
7,3,active_device_key_user_3,device_eshu_s22,2026-04-05 18:55:40.050
8,3,allowed_device_keys_user_3,"[""device_eshu_tablet"",""device_eshu_s22""]",2026-04-05 18:56:15.847
9,3,device_registry_user_3,"[{""backendKey"":""device_bhavi_tab"",""displayName...",2026-04-05 18:55:39.940


In [61]:
cur.execute("""
DELETE FROM settings
WHERE user_id = 3
  AND key = 'tts_enabled_user_3';
""")

cur.execute("""
SELECT user_id, key, value, updated_at
FROM settings
WHERE user_id = 3
  AND key = 'tts_enabled_user_%';
""")

cur.fetchall()



[]

In [69]:
# Sample excute query
cur.execute("""
SELECT user_id, key, value, updated_at
FROM settings
WHERE user_id = 3
  AND key LIKE 'tts_enabled_user_%';


""")

# tables = cur.fetchall()
# pd.DataFrame(tables)
cur.fetchall()

[(3,
  'tts_enabled_user_3',
  '0',
  datetime.datetime(2026, 4, 6, 3, 51, 19, 140000))]

In [9]:
# delete all data from all tables and reset auto-incrementing ids
cur.execute("""
DELETE FROM settings
WHERE user_id = 3
  AND key = 'tts_enabled_user_3';

""")
cur.execute("""commit;""")

In [57]:
# Sample excute query
cur.execute("""
SELECT *
FROM questions
WHERE id IN (31, 110);


""")

tables = cur.fetchall()
pd.DataFrame(tables)

""


In [52]:
# Sample excute query
cur.execute("""
SELECT 
    relname AS table_name,
    n_live_tup AS row_count
FROM pg_stat_user_tables;

""")

tables = cur.fetchall()
pd.DataFrame(tables)

,0,1
0,topics,23
1,badges,0
2,sync_meta,0
3,user_badges,3
4,reviews,0
5,subjects,3
6,settings,12
7,users,3
8,user_subjects,3
9,questions,30


In [ ]:
# table row count
cur.execute("""
SELECT 
    relname AS table_name,
    n_live_tup AS row_count
FROM pg_stat_user_tables;
""")

tables = cur.fetchall()
pd.DataFrame(tables)

,0,1
0,topics,23
1,user_badges,0
2,sync_meta,0
3,badges,0
4,subjects,3
5,user_subjects,3
6,reviews,0
7,users,3
8,stats,0
9,questions,30


In [61]:
# List all tables in the database
cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
""")
cur.fetchall()

[('badges',),
 ('subjects',),
 ('topics',),
 ('questions',),
 ('users',),
 ('user_subjects',),
 ('reviews',),
 ('stats',),
 ('user_badges',),
 ('settings',),
 ('sync_meta',)]

In [9]:
fetch_table_df('users', 50)

,id,name,created_at
0,1,Bhavi,2026-03-30 08:53:00.249952
1,2,Madhu,2026-03-30 08:53:00.249952
2,3,Quiz Kid,2026-03-30 08:53:00.249952


In [10]:
fetch_table_df('subjects', 50)

,id,name
0,1,Mathematics
1,2,English
2,3,Science


In [11]:
# IF parent_topic_id is null, then topic is a subtopic, else it's a main topic
fetch_table_df('topics', 100)

,id,subject_id,parent_topic_id,key,name
0,1,1,NaN,multiplication_tables,Multiplication Tables
1,2,1,1.0,tables_1_5,Tables 1-5
2,3,1,1.0,tables_6_10,Tables 6-10
3,4,1,1.0,tables_11_15,Tables 11-15
4,5,1,1.0,tables_16_20,Tables 16-20
5,6,1,NaN,addition,Addition
6,7,1,NaN,subtraction,Subtraction
7,8,1,NaN,division,Division
8,9,1,NaN,fractions,Fractions
9,10,1,NaN,word_problems,Word Problems


In [12]:
fetch_table_df('questions', 50)

,id,topic_id,type,question,answer
0,1,6,math-addition,4 + 3 = ?,7
1,2,6,math-addition,9 + 6 = ?,15
2,3,7,math-subtraction,12 - 5 = ?,7
3,4,7,math-subtraction,18 - 9 = ?,9
4,5,8,math-division,24 / 6 = ?,4
5,6,8,math-division,35 / 5 = ?,7
6,7,9,fractions,What fraction of 8 slices is 4 slices?,1/2
7,8,9,fractions,What fraction of 10 stars is 5 stars?,1/2
8,9,10,word-problem,Ria has 3 apples and gets 4 more. How many app...,7
9,10,10,word-problem,There are 12 birds and 5 fly away. How many ar...,7


In [ ]:
fetch_table_df('reviews', 30)

In [13]:
fetch_table_df('badges', 50)

""


In [14]:
fetch_table_df('stats', 500)

""


In [15]:
fetch_table_df('settings', 10)

""


In [16]:
fetch_table_df('user_subjects', 10)

,user_id,subject_id
0,1,1
1,2,1
2,3,1


In [17]:
fetch_table_df('user_badges', 5)

""


In [18]:
fetch_table_df('sync_meta', 5)

""


In [80]:
from tests.check_sync_status import main
main()

Centralized Sync Status
----------------------------------------
Overall status : unknown
Overall message: None
Overall updated: Never
--------------------
Push status : unknown
Push message: None
Push updated: Never
--------------------
Pull status : unknown
Pull message: None
Pull updated: Never
--------------------
